# Goal-Aware Expense Forecasting — Inference Demo

Loads the artifact produced by `01_Hybrid_Training.ipynb`, pulls the
selected user's upcoming Google Calendar events, regenerates that user's
ARIMA forecasts, and predicts next month's category-wise expenses with the
trained Random Forest model. Finishes with personalized recommendations.

This notebook performs **inference only** -- no model training happens here.

Requires `hybrid_utils.py` to be present in the same folder as this notebook.

## Imports

In [19]:
import os
import warnings
from datetime import datetime

import joblib
import numpy as np
import pandas as pd

from dateutil.relativedelta import relativedelta
from statsmodels.tsa.arima.model import ARIMA

from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

from hybrid_utils import (
    clean_arima_forecast,
    classify_event_type,
    parse_event_datetime,
    safe_encode_column,
)

warnings.filterwarnings("ignore")

print("Imports OK")

Imports OK


## Configuration

In [20]:
DATA_DIR = "."
USERS_FILE = os.path.join(DATA_DIR, "users.xlsx")
EXPENSES_FILE = os.path.join(DATA_DIR, "monthly_expenses.xlsx")
ARTIFACT_PATH = os.path.join(DATA_DIR, "hybrid_artifacts.pkl")

CREDENTIALS_FILE = os.path.join(DATA_DIR, "credentials.json")
TOKEN_FILE = os.path.join(DATA_DIR, "token.json")
CALENDAR_SCOPES = ["https://www.googleapis.com/auth/calendar.readonly"]
CALENDAR_LOOKAHEAD_MONTHS = 2   # how far ahead to pull calendar events

HISTORY_MONTHS_FOR_ARIMA = 24   # "last 24 months" of history used for forecasting

# Event types considered high / medium importance for inference-time calendar
# features. Training's `event_importance_level` came from a human-labeled
# column in the historical calendar dataset, which live Google Calendar
# events don't have -- these sets are a reasonable inference-time heuristic
# built from the same event-type vocabulary the model was trained on.
HIGH_IMPORTANCE_EVENT_TYPES = {
    "Wedding", "Insurance Renewal", "Rent Due", "EMI Due",
    "College Fee", "School Fee", "Vehicle Purchase",
}
MEDIUM_IMPORTANCE_EVENT_TYPES = {
    "Birthday", "Anniversary", "Festival", "Vacation", "Conference",
    "Business Trip", "Medical Appointment", "Baby Shower", "Housewarming",
    "Exam", "Graduation", "Family Function", "Subscription Renewal", "Parents Visit",
}

print("Configuration loaded")

Configuration loaded


## Step 1 — Load Artifacts & Datasets

In [21]:
def load_inference_artifacts(path):
    """Load the joblib bundle saved by the training notebook and validate
    it contains everything inference needs."""
    bundle = joblib.load(path)

    required_keys = {
        "model", "encoders", "feature_cols", "target_col",
        "categorical_feature_cols", "categories", "base_year",
        "arima_config", "calendar_config",
    }
    missing = required_keys - set(bundle.keys())
    if missing:
        raise KeyError(f"Artifact bundle is missing required keys: {missing}")

    return bundle


def load_datasets(users_path, expenses_path):
    users_df = pd.read_excel(users_path)
    expenses_df = pd.read_excel(expenses_path)
    return users_df, expenses_df


artifacts = load_inference_artifacts(ARTIFACT_PATH)

rf_model = artifacts["model"]
encoders = artifacts["encoders"]
feature_cols = artifacts["feature_cols"]
target_col = artifacts["target_col"]
categories = artifacts["categories"]
base_year = artifacts["base_year"]
arima_config = artifacts["arima_config"]
calendar_config = artifacts["calendar_config"]

users_df, expenses_df = load_datasets(USERS_FILE, EXPENSES_FILE)

print("Artifacts and datasets loaded successfully")
print(f"Model            : {type(rf_model).__name__}")
print(f"Feature count    : {len(feature_cols)}")
print(f"Categories       : {len(categories)}")
print(f"Users            : {users_df.shape}")
print(f"Expenses         : {expenses_df.shape}")

Artifacts and datasets loaded successfully
Model            : RandomForestRegressor
Feature count    : 21
Categories       : 15
Users            : (1000, 10)
Expenses         : (24000, 21)


## Step 2 — Authenticate Google Calendar & Fetch Upcoming Events

In [23]:
def get_calendar_service(credentials_path, token_path, scopes):
    """
    Authenticate against the Google Calendar API using an OAuth
    installed-app flow, caching/refreshing the token locally so the
    browser consent screen only needs to run once.
    """
    creds = None

    if os.path.exists(token_path):
        creds = Credentials.from_authorized_user_file(token_path, scopes)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(credentials_path, scopes)
            creds = flow.run_local_server(port=0)

        with open(token_path, "w") as token_file:
            token_file.write(creds.to_json())

    return build("calendar", "v3", credentials=creds, cache_discovery=False)


def fetch_upcoming_events(service, months_ahead=CALENDAR_LOOKAHEAD_MONTHS, calendar_id="primary"):
    """Fetch all events (single or recurring-expanded) in the next `months_ahead` months."""
    now = datetime.utcnow()
    time_min = now.isoformat() + "Z"
    time_max = (now + relativedelta(months=months_ahead)).isoformat() + "Z"

    try:
        events_result = service.events().list(
            calendarId=calendar_id,
            timeMin=time_min,
            timeMax=time_max,
            singleEvents=True,
            orderBy="startTime",
        ).execute()
    except HttpError as error:
        print(f"Google Calendar API error: {error}")
        return []

    return events_result.get("items", [])

In [24]:
calendar_service = get_calendar_service(CREDENTIALS_FILE, TOKEN_FILE, CALENDAR_SCOPES)
upcoming_events = fetch_upcoming_events(calendar_service)

print(f"Upcoming events found : {len(upcoming_events)}")
for i, event in enumerate(upcoming_events, start=1):
    when = event.get("start", {}).get("dateTime", event.get("start", {}).get("date", "unknown"))
    print(f"  {i}. {event.get('summary', '(no title)')}  --  {when}")

Upcoming events found : 4
  1. EMI Payment  --  2026-08-18T08:00:00+05:30
  2. Fee payment  --  2026-08-20
  3. mom's birthday  --  2026-08-25
  4. EMI Payment  --  2026-09-18T08:00:00+05:30


## Step 3 — Select User & Generate ARIMA Forecasts

In [25]:
def prompt_user_id(users_df):
    """Ask for a user_id in a loop until a valid one is entered."""
    valid_ids = set(users_df["user_id"].astype(str))
    while True:
        user_id = input("Enter Existing User ID (e.g. U00001): ").strip()
        if user_id in valid_ids:
            return user_id
        print(f"User '{user_id}' not found. Please try again.")


def load_user_history(expenses_df, user_id, months=HISTORY_MONTHS_FOR_ARIMA):
    """Load the most recent `months` of monthly expense history for one user."""
    user_rows = expenses_df[expenses_df["user_id"].astype(str) == str(user_id)].copy()
    if user_rows.empty:
        raise ValueError(f"No expense history found for user_id={user_id}")

    user_rows = user_rows.sort_values(["year", "month"]).reset_index(drop=True)
    return user_rows.tail(months).reset_index(drop=True)


def generate_user_arima_forecasts(user_history, categories, arima_config):
    """
    Forecast the next N months for every category using EXACTLY the same
    ARIMA order, minimum-history threshold, forecast horizon, and
    stability bands used during training (all loaded from the artifact).
    """
    order = tuple(arima_config["order"])
    min_history = arima_config["min_history"]
    steps = arima_config["forecast_steps"]
    lower_band = arima_config["lower_band"]
    upper_band = arima_config["upper_band"]

    forecasts = {}
    for category in categories:
        series = user_history[category].astype(float).values if category in user_history.columns else np.array([])
        last_value = float(series[-1]) if len(series) else 0.0

        if len(series) < min_history:
            forecast = np.full(steps, last_value)
        else:
            try:
                model = ARIMA(series, order=order)
                fitted = model.fit()
                forecast = np.asarray(fitted.forecast(steps=steps), dtype=float)
            except Exception:
                forecast = np.full(steps, last_value)

        forecasts[category] = clean_arima_forecast(forecast, last_value, lower_band, upper_band)

    return forecasts

In [26]:
selected_user_id = prompt_user_id(users_df)
user_profile = users_df.loc[users_df["user_id"].astype(str) == selected_user_id].iloc[0]

user_history = load_user_history(expenses_df, selected_user_id)
user_arima_forecasts = generate_user_arima_forecasts(user_history, categories, arima_config)

print("=" * 70)
print("USER PROFILE")
print("=" * 70)
print(f"User ID                : {user_profile['user_id']}")
print(f"Age                    : {user_profile['age']}")
print(f"Gender                 : {user_profile['gender']}")
print(f"Occupation             : {user_profile['occupation']}")
print(f"City                   : {user_profile['city']}")
print(f"Marital Status         : {user_profile['marital_status']}")
print(f"Family Size            : {user_profile['family_size']}")
print(f"Financial Personality  : {user_profile['financial_personality']}")
print(f"\nMonths of history used : {len(user_history)}")
print(f"ARIMA forecasts generated for {len(user_arima_forecasts)} categories")

Enter Existing User ID (e.g. U00001):  U00001


USER PROFILE
User ID                : U00001
Age                    : 24
Gender                 : Male
Occupation             : Driver
City                   : Chennai
Marital Status         : Single
Family Size            : 1
Financial Personality  : Traveller

Months of history used : 24
ARIMA forecasts generated for 15 categories


## Step 4 — Convert Events into Calendar Features (per category)

In [27]:
def infer_event_importance(event_type):
    """Never crashes -- unknown event types default to Low importance."""
    if event_type in HIGH_IMPORTANCE_EVENT_TYPES:
        return "High"
    if event_type in MEDIUM_IMPORTANCE_EVENT_TYPES:
        return "Medium"
    return "Low"


def compute_user_avg_monthly_spend(user_history, categories, window):
    """
    Trailing average TOTAL monthly spend across all categories -- the same
    quantity training's `hist_total` measures (see attach_calendar_features
    in 01_Hybrid_Training.ipynb): a rolling mean of the user's own recent
    spending, clipped to a minimum of 1.0 so early-history / low-spend
    users don't produce an exploding ratio.
    """
    if user_history.empty:
        return 1.0
    recent = user_history.sort_values(["year", "month"]).tail(window)
    total_per_month = recent[categories].sum(axis=1)
    return float(max(total_per_month.mean(), 1.0))


def build_event_features_per_category(events, known_event_types, event_category_map,
                                       default_categories, importance_weight,
                                       categories, user_avg_monthly_spend,
                                       reference_date=None):
    """
    Convert raw Google Calendar events into PER-CATEGORY calendar features,
    matching training's semantics instead of one global feature set
    broadcast to every category row:

      - each event is routed only to the categories EVENT_CATEGORY_MAP says
        it touches (exactly like explode_events_by_category() in training),
        so e.g. a Birthday only affects Food/Entertainment/Shopping and
        Taxi/Public_Transport correctly see event_count=0 for that event.
      - calendar_pressure is a ratio of an approximated event cost against
        the user's own trailing average spend (mirrors training's
        total_estimated_cost / hist_total), not an unrelated flat sum of
        importance weights on a completely different scale.

    Live Calendar events carry no cost field the way the historical
    training data does, so `estimated_total_cost` is approximated as
    importance_weight * the user's own trailing spend baseline -- this
    keeps calendar_pressure on the same scale the model was trained on
    (a small ratio, not an arbitrary sum) even though the exact source
    of the number necessarily differs from training.

    Returns
    -------
    dict[category] -> {event_count, calendar_pressure, planning_months_avg,
                        upcoming_event_type, event_importance_level}
    """
    reference_date = reference_date or pd.Timestamp.utcnow()

    parsed_events = []
    for event in events:
        event_dt = parse_event_datetime(event)
        if event_dt is None:
            continue
        summary = event.get("summary", "") if isinstance(event, dict) else ""
        event_type = classify_event_type(summary, known_event_types)
        importance_level = infer_event_importance(event_type)
        planning_months = max((event_dt - reference_date).days / 30.44, 0.0)
        approx_cost = importance_weight.get(importance_level, 0.5) * user_avg_monthly_spend

        categories_touched = event_category_map.get(event_type, default_categories)
        for category in categories_touched:
            parsed_events.append({
                "expense_category": category,
                "event_type": event_type,
                "event_date": event_dt,
                "planning_months": planning_months,
                "importance_level": importance_level,
                "estimated_cost": approx_cost,
            })

    empty_features = {
        "event_count": 0,
        "calendar_pressure": 0.0,
        "planning_months_avg": 0.0,
        "upcoming_event_type": "None",
        "event_importance_level": "Low",
    }
    features_by_category = {cat: dict(empty_features) for cat in categories}
    if not parsed_events:
        return features_by_category

    events_df = pd.DataFrame(parsed_events)
    baseline = max(user_avg_monthly_spend, 1.0)

    for category, group in events_df.groupby("expense_category"):
        if category not in features_by_category:
            continue
        total_cost = group["estimated_cost"].sum()
        group = group.assign(_w=group["importance_level"].map(importance_weight).fillna(0.5))
        dominant = group.sort_values(["_w", "event_date"], ascending=[False, True]).iloc[0]
        dominant_weight = importance_weight.get(dominant["importance_level"], 0.5)

        features_by_category[category] = {
            "event_count": int(len(group)),
            "calendar_pressure": float((total_cost / baseline) * dominant_weight),
            "planning_months_avg": float(group["planning_months"].mean()),
            "upcoming_event_type": dominant["event_type"],
            "event_importance_level": dominant["importance_level"],
        }

    return features_by_category


In [28]:
known_event_types = list(calendar_config["event_category_map"].keys())

user_avg_monthly_spend = compute_user_avg_monthly_spend(
    user_history, categories, window=calendar_config["trailing_window"]
)

event_features_by_category = build_event_features_per_category(
    upcoming_events, known_event_types, calendar_config["event_category_map"],
    calendar_config["default_event_categories"], calendar_config["importance_weight"],
    categories, user_avg_monthly_spend,
)

print("Calendar features derived from upcoming events (per category):")
for cat in categories:
    ef = event_features_by_category[cat]
    if ef["event_count"] > 0:
        print(f"  {cat:<18}: {ef}")


Calendar features derived from upcoming events (per category):
  Food              : {'event_count': 1, 'calendar_pressure': 1.0, 'planning_months_avg': 0.6241787122207622, 'upcoming_event_type': 'Birthday', 'event_importance_level': 'Medium'}
  Shopping          : {'event_count': 1, 'calendar_pressure': 1.0, 'planning_months_avg': 0.6241787122207622, 'upcoming_event_type': 'Birthday', 'event_importance_level': 'Medium'}
  Entertainment     : {'event_count': 1, 'calendar_pressure': 1.0, 'planning_months_avg': 0.6241787122207622, 'upcoming_event_type': 'Birthday', 'event_importance_level': 'Medium'}
  Miscellaneous     : {'event_count': 3, 'calendar_pressure': 0.75, 'planning_months_avg': 0.7555847568988173, 'upcoming_event_type': 'None', 'event_importance_level': 'Low'}


## Step 5 — Construct Random Forest Input

In [29]:
def compute_time_idx(year, month, base_year):
    return (year - base_year) * 12 + (month - 1)


def determine_next_month(user_history):
    """The month immediately following the user's most recent expense record."""
    last_row = user_history.sort_values(["year", "month"]).iloc[-1]
    if int(last_row["month"]) == 12:
        return int(last_row["year"]) + 1, 1
    return int(last_row["year"]), int(last_row["month"]) + 1


def build_inference_rows(user_id, user_profile, categories, next_year, next_month,
                          next_time_idx, monthly_income, arima_forecasts,
                          event_features_by_category):
    """Build one feature row per expense category, using that category's OWN
    calendar features -- NOT a single value broadcast to every row (see
    build_event_features_per_category in Step 4 for why this matters)."""
    rows = []
    for category in categories:
        forecast = arima_forecasts.get(category, [0.0, 0.0, 0.0])
        ef = event_features_by_category.get(category, {
            "event_count": 0, "calendar_pressure": 0.0,
            "planning_months_avg": 0.0, "upcoming_event_type": "None",
            "event_importance_level": "Low",
        })
        rows.append({
            "user_id": user_id,
            "expense_category": category,
            "time_idx": next_time_idx,
            "monthly_income": monthly_income,
            "year": next_year,
            "month": next_month,
            "age": user_profile["age"],
            "gender": user_profile["gender"],
            "occupation": user_profile["occupation"],
            "city": user_profile["city"],
            "marital_status": user_profile["marital_status"],
            "family_size": user_profile["family_size"],
            "financial_personality": user_profile["financial_personality"],
            "event_count": ef["event_count"],
            "calendar_pressure": ef["calendar_pressure"],
            "planning_months_avg": ef["planning_months_avg"],
            "upcoming_event_type": ef["upcoming_event_type"],
            "event_importance_level": ef["event_importance_level"],
            "arima_month_1": forecast[0],
            "arima_month_2": forecast[1],
            "arima_month_3": forecast[2],
        })
    return pd.DataFrame(rows)


next_year, next_month = determine_next_month(user_history)
next_time_idx = compute_time_idx(next_year, next_month, base_year)
latest_income = float(user_history.iloc[-1]["income"]) if "income" in user_history.columns else float(user_profile["monthly_income"])

raw_input_df = build_inference_rows(
    selected_user_id, user_profile, categories, next_year, next_month,
    next_time_idx, latest_income, user_arima_forecasts, event_features_by_category,
)

print(f"Predicting for : {next_year}-{next_month:02d}  (time_idx={next_time_idx})")
raw_input_df


Predicting for : 2026-01  (time_idx=24)


,user_id,expense_category,time_idx,monthly_income,year,month,age,gender,occupation,city,...,family_size,financial_personality,event_count,calendar_pressure,planning_months_avg,upcoming_event_type,event_importance_level,arima_month_1,arima_month_2,arima_month_3
0,U00001,Food,24,35701.99,2026,1,24,Male,Driver,Chennai,...,1,Traveller,1,1.00,0.624179,Birthday,Medium,7646.509553,7627.418080,7621.715031
1,U00001,Cafe,24,35701.99,2026,1,24,Male,Driver,Chennai,...,1,Traveller,0,0.00,0.000000,None,Low,800.382423,831.362655,821.696040
2,U00001,Taxi,24,35701.99,2026,1,24,Male,Driver,Chennai,...,1,Traveller,0,0.00,0.000000,None,Low,1448.371383,1271.880655,1355.711053
3,U00001,Public_Transport,24,35701.99,2026,1,24,Male,Driver,Chennai,...,1,Traveller,0,0.00,0.000000,None,Low,485.959974,481.757678,482.984020
4,U00001,Shopping,24,35701.99,2026,1,24,Male,Driver,Chennai,...,1,Traveller,1,1.00,0.624179,Birthday,Medium,4988.515491,4799.073207,4743.166601
5,U00001,Entertainment,24,35701.99,2026,1,24,Male,Driver,Chennai,...,1,Traveller,1,1.00,0.624179,Birthday,Medium,1276.674074,1282.324253,1282.202819
6,U00001,Health,24,35701.99,2026,1,24,Male,Driver,Chennai,...,1,Traveller,0,0.00,0.000000,None,Low,1217.039091,1241.763537,1232.343139
7,U00001,Education,24,35701.99,2026,1,24,Male,Driver,Chennai,...,1,Traveller,0,0.00,0.000000,None,Low,2057.096972,2062.209435,2062.259077
8,U00001,Rent,24,35701.99,2026,1,24,Male,Driver,Chennai,...,1,Traveller,0,0.00,0.000000,None,Low,9586.179401,9577.197109,9577.454557
9,U00001,Utilities,24,35701.99,2026,1,24,Male,Driver,Chennai,...,1,Traveller,0,0.00,0.000000,None,Low,1321.063553,1326.251458,1327.135822


## Step 6 — Encode Categorical Variables

In [30]:
def encode_categorical_columns(df, encoders):
    """Encode every categorical column using ONLY the encoders saved at
    training time, safely mapping any unseen label to a default class."""
    encoded = df.copy()
    for col, encoder in encoders.items():
        if col in encoded.columns:
            encoded[col] = safe_encode_column(encoded[col], encoder)
    return encoded


encoded_input_df = encode_categorical_columns(raw_input_df, encoders)

# Feature order comes directly from the artifact -- never hardcoded here.
model_input_df = encoded_input_df[feature_cols]

assert list(model_input_df.columns) == feature_cols, "Feature order mismatch with training artifact"

print("Encoded model input ready")
model_input_df.head()

Encoded model input ready


,user_id,expense_category,time_idx,monthly_income,year,month,age,gender,occupation,city,...,family_size,financial_personality,event_count,calendar_pressure,planning_months_avg,upcoming_event_type,event_importance_level,arima_month_1,arima_month_2,arima_month_3
0,0,3,24,35701.99,2026,1,24,1,6,3,...,1,9,1,1.0,0.624179,2,2,7646.509553,7627.418080,7621.715031
1,0,0,24,35701.99,2026,1,24,1,6,3,...,1,9,0,0.0,0.000000,16,1,800.382423,831.362655,821.696040
2,0,12,24,35701.99,2026,1,24,1,6,3,...,1,9,0,0.0,0.000000,16,1,1448.371383,1271.880655,1355.711053
3,0,9,24,35701.99,2026,1,24,1,6,3,...,1,9,0,0.0,0.000000,16,1,485.959974,481.757678,482.984020
4,0,11,24,35701.99,2026,1,24,1,6,3,...,1,9,1,1.0,0.624179,2,2,4988.515491,4799.073207,4743.166601


## Step 7 — Predict Category-Wise Expenses

In [31]:
def display_predictions(categories, predictions):
    result_df = pd.DataFrame({
        "Expense Category": categories,
        "Predicted Expense": np.round(predictions, 2),
    }).sort_values("Predicted Expense", ascending=False).reset_index(drop=True)

    total_predicted = float(result_df["Predicted Expense"].sum())

    print("=" * 70)
    print("PREDICTED NEXT-MONTH EXPENSES")
    print("=" * 70)
    display(result_df)
    print(f"\nTOTAL PREDICTED EXPENSE : {total_predicted:,.2f}")

    return result_df, total_predicted


predictions = rf_model.predict(model_input_df)
prediction_results_df, total_predicted_expense = display_predictions(categories, predictions)

PREDICTED NEXT-MONTH EXPENSES


,Expense Category,Predicted Expense
0,Food,54811.85
1,Miscellaneous,29823.10
2,Shopping,23698.28
3,Entertainment,15950.50
4,Rent,8529.64
5,Travel,4212.35
6,Education,1919.74
7,Investments,1621.28
8,Health,1234.06
9,Utilities,1207.25



TOTAL PREDICTED EXPENSE : 147,604.73


## Step 8 — Personalized Recommendations

In [32]:
def generate_recommendations(result_df, total_predicted, monthly_income,
                              financial_personality, planning_months_avg,
                              event_importance_level):
    """
    Combine calendar events, predicted expenses, income, financial
    personality, and event planning lead time into savings advice,
    expense-reduction suggestions, and goal-planning suggestions.
    """
    recommendations = []

    projected_savings = monthly_income - total_predicted
    savings_rate = (projected_savings / monthly_income * 100) if monthly_income > 0 else 0.0

    # --- Savings advice ---
    if savings_rate < 10:
        recommendations.append(
            f"Projected savings rate is only {savings_rate:.1f}%. Consider trimming "
            f"discretionary categories before committing to new spending."
        )
    elif savings_rate < 20:
        recommendations.append(
            f"Projected savings rate is {savings_rate:.1f}%, which is reasonable but has "
            f"room to grow -- consider automating a fixed transfer to savings each month."
        )
    else:
        recommendations.append(
            f"Projected savings rate is a healthy {savings_rate:.1f}%. Consider directing "
            f"the surplus toward existing goals or investments."
        )

    # --- Expense reduction suggestions ---
    discretionary = {"Shopping", "Entertainment", "Cafe", "Travel", "Gifts", "Miscellaneous"}
    top_discretionary = (
        result_df[result_df["Expense Category"].isin(discretionary)]
        .sort_values("Predicted Expense", ascending=False)
        .head(2)
    )
    for _, row in top_discretionary.iterrows():
        recommendations.append(
            f"{row['Expense Category']} is projected at {row['Predicted Expense']:,.2f}. "
            f"Setting a soft monthly cap here is one of the fastest ways to free up cash."
        )

    # --- Goal planning suggestions (financial personality) ---
    personality_advice = {
        "Traveller": "Consider a dedicated travel fund and pre-booking flights/stays early to spread the cost.",
        "Family Oriented": "Consider building a family emergency fund covering 3-6 months of expenses.",
        "Saver": "You're well positioned to increase your investment contributions this month.",
        "Spender": "Try setting a category-wise budget alert to keep discretionary spend in check.",
        "Investor": "Review whether this month's surplus can be routed into existing investment SIPs.",
    }
    if financial_personality in personality_advice:
        recommendations.append(personality_advice[financial_personality])

    # --- Calendar-aware goal planning ---
    if event_importance_level in ("Medium", "High"):
        if planning_months_avg < 1:
            recommendations.append(
                "An important upcoming event was detected with very little lead time -- "
                "consider setting aside funds now to avoid a cash crunch."
            )
        else:
            recommendations.append(
                f"An upcoming event was detected roughly {planning_months_avg:.1f} months out. "
                f"Spreading the cost over the next few months should keep it manageable."
            )

    return recommendations


def most_urgent_event_summary(event_features_by_category, importance_rank):
    """
    The recommendations step wants ONE overall event urgency signal, but
    calendar features are now correctly tracked per category (see Step 4).
    Summarize across categories by picking the entry with the highest
    importance (ties broken by soonest planning_months_avg) among
    categories that actually have an upcoming event -- this mirrors what
    the old single global `event_features` dict represented, but is
    derived correctly from the fixed per-category data instead of relying
    on a variable that no longer exists.
    """
    candidates = [
        ef for ef in event_features_by_category.values() if ef["event_count"] > 0
    ]
    if not candidates:
        return {"planning_months_avg": 0.0, "event_importance_level": "Low"}
    return sorted(
        candidates,
        key=lambda ef: (-importance_rank.get(ef["event_importance_level"], 1), ef["planning_months_avg"]),
    )[0]


overall_event_summary = most_urgent_event_summary(
    event_features_by_category, calendar_config["importance_rank"]
)

recommendations = generate_recommendations(
    prediction_results_df,
    total_predicted_expense,
    latest_income,
    user_profile["financial_personality"],
    overall_event_summary["planning_months_avg"],
    overall_event_summary["event_importance_level"],
)

print("=" * 70)
print("PERSONALIZED RECOMMENDATIONS")
print("=" * 70)
for i, rec in enumerate(recommendations, start=1):
    print(f"{i}. {rec}")

PERSONALIZED RECOMMENDATIONS
1. Projected savings rate is only -313.4%. Consider trimming discretionary categories before committing to new spending.
2. Miscellaneous is projected at 29,823.10. Setting a soft monthly cap here is one of the fastest ways to free up cash.
3. Shopping is projected at 23,698.28. Setting a soft monthly cap here is one of the fastest ways to free up cash.
4. Consider a dedicated travel fund and pre-booking flights/stays early to spread the cost.
5. An important upcoming event was detected with very little lead time -- consider setting aside funds now to avoid a cash crunch.
